# Decorator

## TOC

- [Intro](#intro)
- [Decorators with parametres](#decorators-with-parametres)
- [Data in decorators](#data-in-decorators)
- [Decorating methods](#decorating-methods)
- [Classes as decorators](#classes-as-decorators)
- [Decorating classes](#decorating-classes)
- [Wrapping coroutines](#wrapping-coroutines)
- [Docstrings of decorated functions](#docstrings-of-decorated-functions)
- [Summary](#summary)

### Intro

Suppose there is a *foo* [function](https://docs.python.org/3/glossary.html#term-function) that accepts a string and returns its modified version. Imagine that this function is crucial and neither its signature nor body can be changed. In certain cases, the default formatted message must be different, so the task is "to enclose the original message into brackets". Hm, easy-peasy, yet another function that reuses the target (*foo*) function.

In [1]:
def foo(s: str) -> str:
    """Returns a fancy string.

    Args:
        s (str): original string

    Returns:
        str
    """
    return f"string is {s!r}"


def add_brackets_v1(s: str) -> str:
    return f"[{foo(s)}]"


s = 'hello'
print(f"original: {foo(s)}")
print(f"modified: {add_brackets_v1(s)}")

original: string is 'hello'
modified: [string is 'hello']


An ugly solution for `add_brackets` depends on *foo*, so it is just a function that can reuse only the *foo* function and that is all. Ok, my bad, `add_brackets` can be less coupled with *foo* and be used with any string.

In [2]:
def add_brackets_v2(s: str) -> str:
    return f"[{s}]"


s = 'hello'
print(f"original: {foo(s)}")
print(f"modified: {add_brackets_v2(foo(s))}")

original: string is 'hello'
modified: [string is 'hello']


Later, the new wish combinations are asked, so now to maintain four cases:
1. a string in brackets (original -> "[s]")
2. a string in parentheses ("(s)")
3. case 1 and then case 2 ("([s])")
4. case 2 and then case 1 ("[(s)]")

Damn, revise the `add_brackets` and define `add_parentheses` and defi...no, their composition is enough (nested calls)!

In [3]:
def add_brackets(s: str) -> str:
    return f"[{s}]"


def add_parentheses(s: str) -> str:
    return f"({s})"


s = 'hello'
print(f"foo(s): {foo(s)}")
print(f"[foo(s)]: {add_brackets(foo(s))}")
print(f"(foo(s)): {add_parentheses(foo(s))}")
print(f"[(foo(s))]: {add_brackets(add_parentheses(foo(s)))}")
print(f"([foo(s)]): {add_parentheses(add_brackets(foo(s)))}")

foo(s): string is 'hello'
[foo(s)]: [string is 'hello']
(foo(s)): (string is 'hello')
[(foo(s))]: [(string is 'hello')]
([foo(s)]): ([string is 'hello'])


It may seem slick, yet 1) it is static and forms 2) a string of calls which are 3) functional but verbose in a way. Is there an option to reduce this chain of invocations to just one call? In short, yes and decorator patter aims for it. Examples go first and attention to the specially designed functions with a similar structure:

```python
def decorating_function(callable_to_decorate: Callable) -> Callable:
    def wrapper(*args: Any, **kwargs: Any) -> Any:  # so args and kwargs can be any
        # some logic here
        result = callable_to_decorate(*args, **kwargs)
        # some logic here
        # you can return the result
    return wrapper
```

In [4]:
from collections.abc import Callable

# `bracketize` decorator function
def bracketize(func: Callable) -> Callable:
    # `func` is local within `bracketize` decorator (!)
    # `func` is enclosed (!) for `wrapper`
    def wrapper(*args, **kwargs):
        return f"[{func(*args, **kwargs)}]"
    # return a function that can accept any parametres and produce anything
    # every time `wrapper` is called, `func` is also called (closure)
    return wrapper


# `parenthesize` decorator function
def parenthesize(func: Callable) -> Callable:
    # another decorating function `parenthesize` that suits
    # `_` is a valid name for an actual wrapping function
    # because, again, the decorator pattern is also known as the wrapper pattern
    def _(*args, **kwargs):
        return f"({func(*args, **kwargs)})"
    return _


bra = bracketize(foo)
# bra references to the result of `bracketize`
# which is its inner `wrapper` function
# that aceepts *params, **kwparams
# that are given to the wrapped `foo` function
par = parenthesize(foo)

s1 = "tada"
print(s1)

print(bra(s1))  # case 1 - done
print(par(s1))  # case 2 - done

bra_par = bracketize(par)
# par is callable, so `bracketize` can works with it
# it is the same as `bracketize(parenthesize(foo))`
par_bra = parenthesize(bra)

print(bra_par(s1))
print(par_bra(s1))

tada
[string is 'tada']
(string is 'tada')
[(string is 'tada')]
([string is 'tada'])


In Python, [callable](https://docs.python.org/3/glossary.html#term-callable) objects (functions, but not only) can be decorated with `@decorator` syntax. It follows Python's idiomacy when callable objects are good to be decorated statically, so they decorated behaviour is sort of permanent agter the definition.

In [5]:
@parenthesize
# since the inner wrapper is `wrapper(*args, **kwargs)`
# the following function can be called without restrictions
@bracketize
# the order matters
def goo(a: int, b: int) -> float:
    return (a + b) ** .5
# equivalent to `goo = parenthesize(bracketize(goo))`

@bracketize
@parenthesize
def hoo(*args) -> int:
    return len(args)
# equivalent to `hoo = bracketize(parenthesize(hoo))`

print(goo(5, 2))
print(hoo(1, 4, 8))

([2.6457513110645907])
[(3)]


Well, I would like to have the result of `goo` rounded to the 2nd digit after the decimal point. Easy, another decorator.

In [6]:
def round_two(cb: Callable) -> Callable:
    def _deco(*args, **kwargs):
        return float(cb(*args, **kwargs))
    return _deco


@parenthesize
@bracketize
@round_two
def goo_v2(a: int, b: int) -> float:
    # not reusing goo on purpose -> consider this standalone
    return round((a + b) ** .5, 2)

# goo_v2 = parenthesize(bracketize(round_two(goo_v2)))


print(goo(5, 2))
print(goo_v2(5, 2))

([2.6457513110645907])
([2.65])


The major nicety about decorators is that we can change them either statically (via `@deco` sugar) and **dynamically** (`identifier = deco(callable_object)`) which is a flexible way to modify the `callable_object` without actually changing it at all. Decorators are useful when you need to call the well-defined code with additional pre- or post-logic every time the decorated callable is invoked.

### Decorators with parametres

The main parameter (and argument) for a decorating function is a callable object that needs to be wrapped. What if passing any other parameters if this is unavoidable? In the following example on specifying extra parametres does not help and see why:

In [7]:
from typing import Any


# try to remove the default value for `suffix` parametre...aha
def suffux(func: Callable, suffix: str = "") -> Callable:
    def _wrapper(*args, **kwargs):
        return f"{func(*args, **kwargs)} -> {suffix}"
    return _wrapper


def greeter(obj: Any = None) -> str:
    return f"Greetings, {obj}!"


greeter1 = suffux(greeter, "laddie or lassie")
print(greeter1())  # it works, `suffix` parametre is enclosed
print(greeter1("string"))  # prints the expected result


@suffux
def greeter2(obj: Any) -> str:
    return f"Hello, {obj}"


print(greeter2(21))
# and how to change the suffix for the greeter2? :)
# with `greeter1 = suffux(greeter, "laddie or lassie")` it worked
# with greeter2 it is misused!

Greetings, None! -> laddie or lassie
Greetings, string! -> laddie or lassie
Hello, 21 -> 


The situation goes bad if the default argument is not envisaged.

In [8]:
# dammit!
def prefix(func: Callable, prefix: str) -> Callable:
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return f"{prefix} -> {result}"
    return wrapper


# Try uncommenting the following cases


# No way anymore
# @prefix
# def answer() -> int:
#     return 42


# Nice try...not at all :)
# @prefix(prefix="no way")
# def answer() -> int:
#     return 42

# The same as `answer = prefix(prefix="no way")`...shit!
# `func` parametre is required!
# `@prefix(prefix="no way")` is calling the prefix (decorating) function
# and this function requires a Callable at his time which is not the `answer` function

What to do?! Hints before the following solution:
1. you can call a decorator function with an argument -> `@deco(param)`;
2. this `param` can be an enclosed variable in the scope of the `deco` function;
3. calling `@deco(param)` can return a function that expects a function that wraps ...
4. ...
5. maybe PROFIT!

In [9]:
def deco(prefix: str = "prefix", suffix: str = "suffix") -> Callable:
    def decorator(func: Callable) -> Callable:
        def _wrapper(*args, **kwargs):
            result = func(*args, **kwargs)
            return f"{prefix} > {result} < {suffix}"
        return _wrapper
    return decorator
    # `deco("p", "s")` returns `decorator`
    # that can consume a function,
    # i.e., `decorator(func)` returns a `_wrapper`
    # that is called as `_wrapper(*args, **kwargs)`
    # that invokes the wrapped `func`


@deco("H", "T")
def half_answer() -> int:
    return 21
# equivalent form:
# half_answer = deco("H", "T")(half_answer)

print(half_answer())

H > 21 < T


I guess it looks like PROFIT.

Let's see some more action with multiplicator decorator.

In [10]:
def multiplier(coef: float = 1) -> Callable:
    def _deco(func: Callable) -> Callable:
        def _(*args, **kwargs):
            return coef * func(*args, **kwargs)
        return _
    return _deco


@multiplier()
def twelve() -> int:
    return 12


@multiplier(-2)
def twenty_one() -> int:
    return 12


@multiplier(.5)
def fourty_two() -> int:
    return 42


print(twelve())  # yes
print(twenty_one())  # not exactly -> -42
print(fourty_two())  # twisted -> 21.0


# even like this
invariant = multiplier(-.5)(multiplier(-2)(lambda: 10))
print(invariant())

12
-24
21.0
10.0


### Data in decorators

There are situation when decorators should have data stored under the hood. If these data are mutable, risks of changing them unexpectedly are viable. Consider the example on a decorator that tracks function call count and also has a small cache for return results in order to add some performance via [memoisation](https://en.wikipedia.org/wiki/Memoization) technique.

In [11]:
def memoize(func: Callable) -> Callable:
    def _wrapper(*args, **kwargs):
        # this implementation is not saviour
        cache: dict[str, Any] = {}
        kws = {k:kwargs[k] for k in sorted(kwargs)}
        if (params := f"{sorted(args)}:{kws}") in cache:
            print(f"{func.__name__}: result from cache")
            return cache[params]
        res = func(*args, **kwargs)
        cache[params] = res
        return res
    return _wrapper


@memoize
def divide(a: int, b: int) -> tuple[int, int]:
    return divmod(a, b)


divide(13, 4)
divide(13, 4)

(3, 1)

This works, but not as expected. Every time the wrapped function is called, the _cache_ local varaible is initialised. When the *_wrapper* scope is left, the cache is gone. Let's move the cache instantiation out of the *_wrapper* function.

In [12]:
def memoize(func: Callable) -> Callable:
    cache: dict[str, Any] = {}
    def _wrapper(*args, **kwargs):
        kws = {k:kwargs[k] for k in sorted(kwargs)}
        if (params := f"{sorted(args)}:{kws}") in cache:
            print(f"{func.__name__}: result from cache")
            return cache[params]
        res = func(*args, **kwargs)
        cache[params] = res
        return res
    return _wrapper


@memoize
def divide(a: int, b: int) -> tuple[int, int]:
    return divmod(a, b)


divide(13, 4)
divide(13, 4)  # from cache

### looks good

@memoize
def answer() -> int:
    return 42

answer()
divide(0, 6)
answer()  # from cache

divide: result from cache
answer: result from cache


42

Now it is better. Every time a function (callable object), shall we say *foo*, is wrapped with the *memoize* decorator, the *cache* is initialised at the time of `foo = memoize(foo)` (or `@memoize` sugar) andv it stays well defined and managed for a particular wrapped function. You can guess what may go wrong if a cache would have been defined globally outside the decorator. The example above is easy to change to see the illustrative effect.

### Decorating methods

[Method](https://docs.python.org/3/glossary.html#term-method)s are callable objects associated with a class. The first parametre/argument for a method is either a reference to an instance of this class (*self*) or a class (*cls*). The next example is OK regardless TypeError.

In [13]:
def decoco(func: Callable[[int, int], int]) -> Callable[[int, int], str]:
    def _(a: int, b: int) -> str:
        return f"result = {func(a, b)}"
    return _


@decoco
def add(x: int, y: int) -> int:
    return x + y


print(add(3, 7))  # OK


class Adder:
    @decoco  # signature mismatch
    def add(self, x: int, y: int) -> int:
        return x + y


print(Adder().add(3, 7))

result = 10


TypeError: decoco.<locals>._() takes 2 positional arguments but 3 were given

The first argument is for self or cls object, so the first argument is associated with the self|cls parametre and the second argument is bound to the *x* parametre, so the *y* parametre is not given, but required. Let's try with variadic params and keyword params.

In [14]:
def decoqo(func: Callable[..., int]) -> Callable[..., str]:
    def _(*params, **kwparams) -> str:
        return f"result = {func(*params, **kwparams)}"
    return _


@decoqo
def add(x: int, y: int) -> int:
    return x + y


print(add(3, 7))  # OK


class Adder:
    @decoqo  # whatever signature
    def add(self, x: int, y: int) -> int:
        return x + y

# TypeError: decoco.<locals>._() takes 2 positional arguments but 3 were given
print(Adder().add(3, 7))

result = 10
result = 10


### Classes as decorators

Not only functions can decorate functions/methods. Again, a decorator is a callable object that accepts another callable object. An object is callable if it can be used with parentheses where zero or more (kw)params can be placed, i.e., `obj(*args, kwargs)`. An instance of a class is callable if it defines the [\_\_call\_\_()](https://docs.python.org/3/reference/datamodel.html#object.__call__) dunder. A callable object is tested via the [callable](https://docs.python.org/3/library/functions.html#callable) builtin.

In [15]:
def sumargs(*args, **kwargs):
    return sum([len(args), len(kwargs)])

assert callable(sumargs)
# foo(...) is foo.__call__(...)
assert sumargs(1, 2, a='b') == sumargs.__call__(1, 2, a='b')


class Dummy:
    def __init__(self) -> None:
        self.c = []


assert not callable(Dummy())  # no __call__ dunder


class DummyCallable:
    def __init__(self) -> None:
        self._c: list = []

    def items(self) -> list:
        return self._c[:]

    def __call__(self, *args: Any) -> None:
        self._c.extend(args)


dc = DummyCallable()
assert callable(dc)

dc(1, 2)
dc.__call__(-4, 3)

assert dc.items() == [1, 2, -4, 3]

Time for a class that is callable and designed to decorate other callables.

In [16]:
class Decorator:
    def __init__(self, limit: int = 5) -> None:
        if limit < 0:
            raise ValueError("limit < 0")
        self._limit = limit
        self._func: Callable  # must be defined, e.g. in the __call__

    def _wrapper(self, *args, **kwargs):
        if not self._limit:
            raise ValueError("call limit exceeded")
        res = self._func(*args, **kwargs)
        self._limit -= 1
        return res

    def __call__(self, func: Callable) -> Any:
        # or you can define a wrapper here as an ordinary function
        self._func = func
        return self._wrapper


@Decorator(limit=2)
def power(base: float, exp: float) -> float:
    return base ** exp

# power = Decorator(limit=2)(power)

print(power(2, 4))
print(power(3, 2))
print(power(1, 1))

16
9


ValueError: call limit exceeded

### Decorating classes

We know that functions are definitely callable objects, but what about wrapping classes? Can we decorate them and if so, is there more than one way to do so? Let's just try.

In [17]:
def plog(func: Callable) -> Callable:
    def _(*args: Any, **kwargs: Any) -> Any:
        res =  func(*args, **kwargs)
        print(f"{func.__name__} -> {res}")
        return res
    return _


@plog
class Dummy:
    def __init__(self, number: float) -> None:
        self._nbr = number

    def __repr__(self) -> str:
        return f"{type(self).__name__}({self._nbr})"

    @property
    def number(self) -> float:
        return self._nbr

    def scale(self, coef: float) -> float:
        return self.number * coef

dummy = Dummy(5)  # only here the decorator works
print()
print(f"scaled by -2: {dummy.scale(-2)}")  # no effect
print(f"number = {dummy.number}")  # no plogging is done

Dummy -> Dummy(5)

scaled by -2: -10
number = 5


So, the decorating logic works only when the class is instantiated and is not in effect for other methods even dunders.

In fact, the classes are callable. From the [docs](https://docs.python.org/3/reference/datamodel.html#classes):
> Classes are callable. These objects normally act as factories for new instances of themselves, but variations are possible for class types that override [\_\_new\_\_()](https://docs.python.org/3/reference/datamodel.html#object.__new__). The arguments of the call are passed to \_\_new\_\_() and, in the typical case, to [\_\_init\_\_()](https://docs.python.org/3/reference/datamodel.html#object.__init__) to initialize the new instance.

A [quote](https://docs.python.org/3/reference/datamodel.html#class-instances) about class instances:
> Instances of arbitrary classes can be made callable by defining a \_\_call\_\_() method in their class.

So, wrappping a class is like wrapping its \_\_init\_\_() dunder. It makes sense because calling a class means to create and return its instance.

### Wrapping coroutines

Coroutines are easy to wrap yet some points to remember:
- `await` inside a sync `def` function is SyntaxError
- so, a wrapper should be an `async def` function

In [18]:
def wrap_coro(coro: Callable) -> Callable:
    async def _wrapper(*args, **kwargs):
        print(f"before {coro}")
        result = await coro(*args, **kwargs)
        print(f"after {coro} -> {result}")
        return result
    return _wrapper


@wrap_coro
async def coro(*args):
    print(f"I am just coroutine with {args}")


await coro()

before <function coro at 0x756bdadd4e00>
I am just coroutine with ()
after <function coro at 0x756bdadd4e00> -> None


One decorator for wrapping either a routine or a coroutine? It is doable and [this solution](https://stackoverflow.com/questions/44169998/how-to-create-a-python-decorator-that-can-wrap-either-coroutine-or-function) looks good to me yet it may be a step to overengineering.

In [19]:
import inspect
from contextlib import contextmanager


def powerful_decorator(func):
    @contextmanager
    def wrapping_logic():
        print("Hello")
        yield
        print("Bye")

    def wrapper(*args, **kwargs):
        if not inspect.iscoroutinefunction(func):
            with wrapping_logic():
                return func(*args, **kwargs)
        print("Wrapping a coroutine")
        async def tmp():
            with wrapping_logic():
                return (await func(*args, **kwargs))
        return tmp()

    return wrapper


@powerful_decorator
def synco():
    print("Time to sync")


@powerful_decorator
async def asynco():
    print("Our sync is run async")


synco()
await asynco()

Hello
Time to sync
Bye
Wrapping a coroutine
Hello
Our sync is run async
Bye


### Docstrings of decorated functions

So far the things go so good. Writing decorators is fun, we can decorate functions, methods, classes yet they were undocumented and documenting public entities is what should be done for maintainable code's sake. So, what can go wrong if a docstring is just a docstring and not a big deal to add it.

In [20]:
def docstring_perdu(cb: Callable) -> Callable:
    def _wrapper(*args, **kwargs):
        return cb(*args, **kwargs)
    return _wrapper


def divide_without_conquering(a: float, b: float) -> float:
    """Perform `a / b` logic.

    Args:
        a (float): dividend
        b (float): divisor

    Raises:
        ZeroDivisionError

    Returns:
        int
    """
    return a / b


print(divide_without_conquering.__doc__)  # OK

decorated = docstring_perdu(divide_without_conquering)
print(decorated.__doc__)  # What ?!

Perform `a / b` logic.

    Args:
        a (float): dividend
        b (float): divisor

    Raises:
        ZeroDivisionError

    Returns:
        int
    
None


Docstring is lost because *_wrapper* does not have it. The idea of specifying a docstring in the *_wrapper* is of no use:

- the wrapped object must not be stripped of its docstring, obviously;
- docstring in the *_wrapper* is duplication;
- what will happen if the same decorator wraps different functions with different docstring?

Gladly, thre is much more fancy solution -> [functools.wraps](https://docs.python.org/3/library/functools.html).


In [21]:
import functools

def docstring_saved(cb: Callable) -> Callable:
    @functools.wraps(cb)  # accepts the target cb and wraps the wrapper
    def _wrapper(*args, **kwargs):
        return cb(*args, **kwargs)
    return _wrapper


@docstring_saved
def divide_without_conquering(a: float, b: float) -> float:
    """Perform `a / b` logic.

    Args:
        a (float): dividend
        b (float): divisor

    Raises:
        ZeroDivisionError

    Returns:
        int
    """
    return a / b


print(divide_without_conquering.__doc__)  # OK

Perform `a / b` logic.

    Args:
        a (float): dividend
        b (float): divisor

    Raises:
        ZeroDivisionError

    Returns:
        int
    


Nicely done.

### Summary

Honestly, I thought that [Decorator](https://en.wikipedia.org/wiki/Decorator_pattern), also referred to as Wrapper, is a *behavioural* design pattern for we can change behaviour of a decorated function or class and stuff. Yet this pattern is **structural** and structural patterns focus on the composition and (re)structuring that is why such name. The fact that Decorator/Wrapper is structural has grounded reasons because:

1. The decorated callable does not change neither in structure not behaviour.
2. A wrapper over it uses the wrappee as a component which means composition (preferred to inheritance).
3. The resultant object expresses the relationship between the decorator and wrappee which is modified structure.

That is all for now.